In [ ]:
# Load packages
#---------------------------------------------------------------------

import pandas as pd
import numpy as np

import sys
import os
sys.path.append(os.path.abspath('..'))

from config import *

# Load data
#---------------------------------------------------------------------

df = pd.read_csv('../data/raw/realized_var_stocks.csv')

# Filter to selected stocks from config
df = df[df['symbol'].isin(stocks)].copy()

# Feature engineering
#---------------------------------------------------------------------

def create_features(data):
    """
    Generate features for all six forecasting models.

    All rolling statistics, lags, and shifts are computed within each stock
    group separately to avoid information leakage across symbols.

    Feature map
    -----------
    GJR-GARCH  : ret  (open-to-close returns passed directly to arch_model)
    MF2-GARCH  : ret  (open-to-close returns passed directly to MF2GARCH.fit)
    HAR-GARCH  : log_rv_d, log_rv_w, log_rv_m, ret_neg  (HAR_COLS in har_garch.py)
    XGBoost    : log_rv_d, log_rv_w, log_rv_m,
                 rv_lag_5, rv_lag_22, rv_ratio, ret  (xgb_features from config)
    MLP        : log_rv_d, log_rv_w, log_rv_m, ret  (mlp_features from config)
    Hybrid     : xgb_features + mf2_h, mf2_tau      (added inside HybridModel.fit)

    Intermediate columns rv_roll_5, rv_roll_22 are computed to build log_rv_w/m
    and rv_ratio but are not passed to any model directly.
    """

    data = data.copy()
    grouped = data.groupby('symbol')

    # --- Returns ---
    # Open-to-close log-return, scaled by 100:  ret_t = 100 * log(close_t / open_t).
    # This spans exactly the regular trading session (09:30-16:05) over which VOLARE
    # computes rv5, so the GARCH-family models (GJR-GARCH, MF2-GARCH, and the
    # MF2 component inside the Hybrid) model the conditional variance of the SAME
    # intraday quantity that rv5 measures. A close-to-close return would additionally
    # contain the overnight move, which rv5 does not capture, leaving those forecasts
    # on a systematically larger scale than the evaluation target.
    data['ret'] = 100 * (np.log(data['close_price']) - np.log(data['open_price']))

    # --- Leverage term: ret_neg = min(ret_t, 0) for HAR-GARCH only  ---
    # ret_neg isolates the leverage channel (negative returns raise future volatility; 
    # positive ones leave the forecast unchanged) -- the LHAR term of Corsi & Reno (2012),
    # and the linear analogue of the GJR/MF2 gamma*I{r<0} asymmetry.
    data['ret_neg'] = np.minimum(data['ret'], 0.0)

    # --- Realized Variance (RV) ---
    data['rv'] = data['rv5']

    # Point RV lags
    for lag in [5, 22]:
        data[f'rv_lag_{lag}'] = grouped['rv'].transform(lambda x: x.shift(lag))

    # Rolling means ENDING at day t (they include today's RV) - standard HAR timing
    for roll in [5, 22]:
        data[f'rv_roll_{roll}'] = grouped['rv'].transform(lambda x: x.rolling(roll).mean())

    # --- Log-RV HAR components ---
    # Standard HAR timing: daily = RV_t, weekly = mean(RV_{t-4..t}), monthly = mean(RV_{t-21..t})
    # each component INCLUDES the current day's RV. Clip before log to avoid -inf on zero or near-zero values.
    data['log_rv']   = np.log(data['rv'].clip(lower=1e-10))
    data['log_rv_d'] = np.log(data['rv'].clip(lower=1e-10))
    data['log_rv_w'] = np.log(data['rv_roll_5'].clip(lower=1e-10))
    data['log_rv_m'] = np.log(data['rv_roll_22'].clip(lower=1e-10))

    # --- Short/long RV ratio — regime signal ---
    data['rv_ratio'] = data['rv_roll_5'] / data['rv_roll_22'].clip(lower=1e-10)

    # --- Forecast targets ---
    # target_rv_1  : next-day RV
    # target_rv_5  : mean RV over the next 5 days  (t+1 ... t+5)
    # target_rv_22 : mean RV over the next 22 days (t+1 ... t+22)
    data['target_rv_1']  = grouped['rv'].transform(lambda x: x.shift(-1))
    data['target_rv_5']  = grouped['rv'].transform(lambda x: x.shift(-5).rolling(5).mean())
    data['target_rv_22'] = grouped['rv'].transform(lambda x: x.shift(-22).rolling(22).mean())

    data = data.dropna()

    return data

df = create_features(df)

# Save processed features
df.to_excel('../data/processed/rv_features_created.xlsx', index=False)